<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one report-date observation for one pseudonymized client and one pseudonymized content item.

**Primary table:** `fact_content_daily_performance`.

**Development time window:** March 2026 (`month=2026-03`).

I will use March 2026 as the mid-panel development month. I will not use the final-month `_sample` for developing label logic because it represents the latest full month and will be treated as a sealed test period.

**Prediction/ranking goal:** I want to rank content pages by their likelihood of showing declining search performance, using only information available before the decision moment.

**Deliberate exclusion:** I will exclude fields derived from the future outcome or fields that would only be available after the decision, because they could introduce target leakage.

In [1]:
from google.colab import userdata
from huggingface_hub import HfApi, HfFileSystem

HF_TOKEN = userdata.get("HF_TOKEN")

# Test that the token can access the dataset
api = HfApi(token=HF_TOKEN)
info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset access confirmed:", info.id)

Dataset access confirmed: FlyRank/internship-warehouse


In [2]:
!pip -q install -U huggingface_hub duckdb
from huggingface_hub import HfFileSystem
import duckdb

# Create filesystem using your authenticated token
fs = HfFileSystem(token=HF_TOKEN)

# Register the authenticated filesystem with DuckDB
duckdb.register_filesystem(fs)

print("Authenticated Hugging Face filesystem registered.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 78.7 MB/s eta 0:00:00
Authenticated Hugging Face filesystem registered.


In [3]:
files = fs.glob(
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Number of March 2026 files:", len(files))
print(files[:5])

Number of March 2026 files: 1
['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']


In [4]:
con = duckdb.connect()

print("New DuckDB connection created.")

New DuckDB connection created.


In [9]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

files = list(
    api.list_repo_tree(
        repo_id="FlyRank/internship-warehouse",
        path_in_repo="fact_content_daily_performance/month=2026-03",
        repo_type="dataset",
        recursive=False
    )
)

for f in files:
    print(f.path)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [10]:
for f in files:
    print(f.path, "→", f.size / (1024**3), "GB")

fact_content_daily_performance/month=2026-03/data_0.parquet → 0.11570165865123272 GB


In [18]:
try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("✅ Dataset access confirmed")
    print(info.id)
except Exception as e:
    print("❌ Dataset access problem")
    print(type(e).__name__, str(e))

✅ Dataset access confirmed
FlyRank/internship-warehouse


In [24]:
from huggingface_hub import hf_hub_download

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded successfully:")
print(local_file)

GatedRepoError: 401 Client Error. (Request ID: Root=1-6a7cb8a9-7853ff39027dc3634de0a467;b5488a08-6273-4169-8568-36668db4f9a1)

Cannot access gated repo for url https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month%3D2026-03/data_0.parquet.
Access to dataset FlyRank/internship-warehouse is restricted. You must have access to it and be authenticated to access it. Please log in.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features

I will use historical search-performance variables that were available before the decision moment. I will keep the feature set to a maximum of five variables.

### Label

The label/proxy will represent a future decline in search performance. It will be defined from an outcome window after the feature/decision window.

### Context

Pseudonymized client and content identifiers will be used for grouping, joining, and verification rather than treated as meaningful predictive signals.

### Excluded

I will exclude fields that directly encode the outcome, are calculated from the future window, or would only be known after the decision. These fields could cause target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
grain_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (
            report_date,
            client_id,
            content_id
        )) AS distinct_grain
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    """
).df()

grain_check

HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet' (HTTP 0 Internal Server Error)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



### Limitation: Unequal historical coverage

The warehouse is an unbalanced panel, so different clients have different depths of historical data. This means that observations from different clients may not represent equally complete histories.

The data is observational rather than experimental. It can show measured patterns and support ranking decisions, but it cannot by itself prove that a particular content change caused a future improvement in search performance.

The fixed 90-day query windows can also overlap reporting periods, so feature and outcome windows must be defined carefully to avoid using future information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.